# 📊 Notebook 01 — Exploratory Data Analysis (EDA)
## MarocChurn · Telecom Customer Churn Prediction
**Author:** Ahmed Mansof · Master Data Science · ENS Tétouan

---

### 🎯 Objectives
1. Understand the dataset structure and variable types
2. Analyze the distribution of each feature
3. Detect missing values and outliers
4. Study the relationship between each feature and churn
5. Extract key business insights to guide modelling

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1e2d',
    'axes.facecolor':   '#0d1e2d',
    'axes.edgecolor':   '#1a3348',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#1a3348',
    'grid.alpha':       0.6,
    'axes.titlecolor':  'white',
    'axes.titlesize':   13,
    'axes.titlepad':    12,
})

CYAN   = '#00e5ff'
PURPLE = '#7b61ff'
ORANGE = '#ff6b35'
MUTED  = '#6b8fa3'

print("✅ Libraries imported successfully")

## 1. Load & Inspect the Dataset

In [ ]:
# ── Load data ────────────────────────────────────────────────
import os

DATA_PATH = '../data/raw/telco_churn.csv'

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    df.columns = df.columns.str.strip()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    print(f"✅ Real dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
else:
    # ── Synthetic data (if Kaggle CSV not available) ──────────
    np.random.seed(42)
    n = 7043
    tenures  = np.random.randint(0, 73, n)
    monthly  = np.round(np.random.uniform(18, 119, n), 2)
    contracts= np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55,0.24,0.21])
    churn_p  = np.clip(0.05 + 0.35*(contracts=='Month-to-month') + 0.20*(tenures<6) + 0.15*(monthly>80), 0, 1)
    churn    = np.array(['Yes' if np.random.random()<p else 'No' for p in churn_p])
    df = pd.DataFrame({
        'customerID': [f'C{i:05d}' for i in range(n)],
        'gender': np.random.choice(['Male','Female'], n),
        'SeniorCitizen': np.random.choice([0,1], n, p=[0.84,0.16]),
        'Partner': np.random.choice(['Yes','No'], n),
        'Dependents': np.random.choice(['Yes','No'], n, p=[0.3,0.7]),
        'tenure': tenures,
        'PhoneService': np.random.choice(['Yes','No'], n, p=[0.9,0.1]),
        'MultipleLines': np.random.choice(['Yes','No','No phone service'], n),
        'InternetService': np.random.choice(['Fiber optic','DSL','No'], n, p=[0.44,0.34,0.22]),
        'OnlineSecurity': np.random.choice(['Yes','No','No internet service'], n),
        'OnlineBackup': np.random.choice(['Yes','No','No internet service'], n),
        'DeviceProtection': np.random.choice(['Yes','No','No internet service'], n),
        'TechSupport': np.random.choice(['Yes','No','No internet service'], n),
        'StreamingTV': np.random.choice(['Yes','No','No internet service'], n),
        'StreamingMovies': np.random.choice(['Yes','No','No internet service'], n),
        'Contract': contracts,
        'PaperlessBilling': np.random.choice(['Yes','No'], n, p=[0.59,0.41]),
        'PaymentMethod': np.random.choice([
            'Electronic check','Mailed check',
            'Bank transfer (automatic)','Credit card (automatic)'], n),
        'MonthlyCharges': monthly,
        'TotalCharges': np.round(tenures*monthly + np.random.uniform(-50,50,n), 2),
        'Churn': churn
    })
    print(f"⚠️  Using synthetic data: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("   → Download real data from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn")

print(f"\nColumns: {list(df.columns)}")

In [ ]:
# ── Basic overview ────────────────────────────────────────────
print("=" * 55)
print(" DATASET OVERVIEW")
print("=" * 55)
print(f"  Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Memory usage  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"  Churn rate    : {(df['Churn']=='Yes').mean()*100:.1f}%")
print(f"  Missing vals  : {df.isnull().sum().sum()}")
print("=" * 55)
df.head(5)

In [ ]:
# ── Data types & missing values ──────────────────────────────
info_df = pd.DataFrame({
    'Type':     df.dtypes.astype(str),
    'Non-Null': df.notnull().sum(),
    'Nulls':    df.isnull().sum(),
    'Null %':   (df.isnull().sum() / len(df) * 100).round(2),
    'Unique':   df.nunique()
})
print("\n📋 Column info:")
display(info_df)

## 2. Target Variable — Churn Distribution

In [ ]:
# ── Churn distribution ───────────────────────────────────────
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
colors = [CYAN, PURPLE]
wedges, texts, autotexts = axes[0].pie(
    churn_counts, labels=churn_counts.index,
    autopct='%1.1f%%', colors=colors, startangle=90,
    textprops={'color':'white', 'fontsize':12},
    wedgeprops={'edgecolor': '#060d16', 'linewidth': 2}
)
for at in autotexts: at.set_fontsize(13); at.set_fontweight('bold')
axes[0].set_title('Churn Distribution (Pie)', pad=15)

# Bar chart
bars = axes[1].bar(churn_counts.index, churn_counts.values,
                   color=[CYAN, PURPLE], edgecolor='none', width=0.5)
for bar, pct in zip(bars, churn_pct):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 30,
                 f'{pct:.1f}%\n({bar.get_height():,})',
                 ha='center', color='white', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Churn Count (Bar)')
axes[1].set_ylim(0, churn_counts.max() * 1.2)

fig.suptitle('⚠️  Class Imbalance: ~26.5% Churn (will handle with SMOTE)',
             color=ORANGE, fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/plot_churn_distribution.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()
print(f"\nChurn : {churn_counts['Yes']:,} customers ({churn_pct['Yes']:.1f}%)")
print(f"No Churn: {churn_counts['No']:,} customers ({churn_pct['No']:.1f}%)")

## 3. Numerical Features Distribution

In [ ]:
# ── Distributions of numerical variables ─────────────────────
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
churned     = df[df['Churn']=='Yes']
not_churned = df[df['Churn']=='No']

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('Numerical Features: Distribution vs Churn', fontsize=14, y=1.01)

for i, col in enumerate(num_cols):
    # Left: KDE comparison
    ax = axes[i][0]
    ax.hist(not_churned[col].dropna(), bins=35, alpha=0.7,
            color=CYAN, label='No Churn', density=True, edgecolor='none')
    ax.hist(churned[col].dropna(), bins=35, alpha=0.7,
            color=ORANGE, label='Churn', density=True, edgecolor='none')
    ax.set_title(f'{col} — KDE by Churn')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend(facecolor='#0d1e2d', edgecolor='#1a3348')

    # Right: Boxplot
    ax2 = axes[i][1]
    data_plot = [not_churned[col].dropna(), churned[col].dropna()]
    bp = ax2.boxplot(data_plot, patch_artist=True, notch=True,
                      labels=['No Churn', 'Churn'],
                      medianprops=dict(color='white', linewidth=2))
    bp['boxes'][0].set_facecolor(CYAN + '88')
    bp['boxes'][1].set_facecolor(ORANGE + '88')
    for element in ['whiskers','caps','fliers']:
        for item in bp[element]: item.set_color(MUTED)
    ax2.set_title(f'{col} — Boxplot by Churn')
    ax2.set_ylabel(col)

plt.tight_layout()
plt.savefig('../data/processed/plot_numerical_distributions.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

# Stats
print("\n📊 Key Statistics by Churn:")
print(df.groupby('Churn')[num_cols].mean().round(2).to_string())

## 4. Categorical Features vs Churn

In [ ]:
# ── Churn rate per categorical feature ───────────────────────
cat_features = ['Contract', 'InternetService', 'PaymentMethod',
                'gender', 'SeniorCitizen', 'Partner', 'Dependents',
                'PaperlessBilling', 'PhoneService', 'TechSupport',
                'OnlineSecurity', 'StreamingTV']

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle('Churn Rate by Categorical Feature', fontsize=14, y=1.01)
axes = axes.flatten()

for i, col in enumerate(cat_features):
    ax = axes[i]
    ct = df.groupby(col)['Churn'].apply(lambda x: (x=='Yes').mean()*100)
    ct = ct.sort_values(ascending=False)
    colors_bar = [ORANGE if v > 30 else (CYAN if v < 15 else PURPLE)
                  for v in ct.values]
    bars = ax.barh(ct.index.astype(str), ct.values, color=colors_bar, edgecolor='none')
    for bar, val in zip(bars, ct.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8, color='white')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Churn Rate (%)')
    ax.set_xlim(0, ct.max() * 1.35)
    ax.axvline(x=df['Churn'].eq('Yes').mean()*100,
               color='white', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('../data/processed/plot_categorical_vs_churn.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

## 5. Correlation Heatmap

In [ ]:
# ── Correlation matrix ───────────────────────────────────────
# Encode for correlation
df_enc = df.copy()
df_enc['Churn_bin'] = (df_enc['Churn'] == 'Yes').astype(int)
for col in df_enc.select_dtypes('object').columns:
    from sklearn.preprocessing import LabelEncoder
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
             'Contract', 'InternetService', 'TechSupport', 'OnlineSecurity',
             'PaperlessBilling', 'Churn_bin']
corr = df_enc[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, linecolor='#060d16',
            annot_kws={'size': 9, 'color': 'white'},
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — All Features', fontsize=14, pad=15)
ax.tick_params(labelsize=9, rotation=45)
plt.tight_layout()
plt.savefig('../data/processed/plot_correlation_heatmap.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

# Top correlations with churn
print("\n📊 Top correlations with Churn:")
churn_corr = corr['Churn_bin'].drop('Churn_bin').abs().sort_values(ascending=False)
for feat, val in churn_corr.head(8).items():
    print(f"  {feat:<25} {val:.3f}")

## 6. Tenure Analysis — The Key Retention Signal

In [ ]:
# ── Tenure buckets vs churn ──────────────────────────────────
df['tenure_group'] = pd.cut(df['tenure'],
    bins=[0, 12, 24, 36, 48, 60, 72],
    labels=['0-12m', '13-24m', '25-36m', '37-48m', '49-60m', '61-72m'])

tenure_churn = df.groupby('tenure_group', observed=True)['Churn'].apply(
    lambda x: (x=='Yes').mean()*100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by tenure group
bars = axes[0].bar(tenure_churn.index.astype(str), tenure_churn.values,
                   color=[ORANGE if v > 30 else (PURPLE if v > 15 else CYAN)
                          for v in tenure_churn.values],
                   edgecolor='none')
for bar, val in zip(bars, tenure_churn.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', color='white', fontsize=10, fontweight='bold')
axes[0].set_title('Churn Rate by Tenure Group')
axes[0].set_xlabel('Tenure Group')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, tenure_churn.max() * 1.2)

# Monthly charges vs tenure colored by churn
sample = df.sample(min(2000, len(df)), random_state=42)
churn_map = {'Yes': ORANGE, 'No': CYAN}
for churn_val, color, label in [('No', CYAN, 'No Churn'), ('Yes', ORANGE, 'Churn')]:
    sub = sample[sample['Churn'] == churn_val]
    axes[1].scatter(sub['tenure'], sub['MonthlyCharges'],
                    c=color, alpha=0.3, s=15, label=label)
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Monthly Charges ($)')
axes[1].set_title('Tenure vs Monthly Charges (colored by Churn)')
axes[1].legend(facecolor='#0d1e2d', edgecolor='#1a3348')

plt.tight_layout()
plt.savefig('../data/processed/plot_tenure_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1e2d')
plt.show()

## 7. EDA Summary & Key Insights

In [ ]:
# ── Key insights ─────────────────────────────────────────────
print("=" * 60)
print(" 🔍 KEY INSIGHTS FROM EDA")
print("=" * 60)

# Insight 1: Contract
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x=='Yes').mean()*100)
print(f"\n1. CONTRACT TYPE is the strongest predictor:")
for k, v in contract_churn.sort_values(ascending=False).items():
    print(f"   {k:<20} → {v:.1f}% churn rate")

# Insight 2: New customers
new = df[df['tenure'] <= 12]['Churn'].eq('Yes').mean() * 100
old = df[df['tenure'] > 48]['Churn'].eq('Yes').mean() * 100
print(f"\n2. TENURE effect:")
print(f"   New customers (≤12m) → {new:.1f}% churn")
print(f"   Loyal customers (>48m) → {old:.1f}% churn")

# Insight 3: Internet
internet_churn = df.groupby('InternetService')['Churn'].apply(
    lambda x: (x=='Yes').mean()*100)
print(f"\n3. INTERNET SERVICE:")
for k, v in internet_churn.sort_values(ascending=False).items():
    print(f"   {k:<20} → {v:.1f}% churn rate")

# Insight 4: Missing values
print(f"\n4. MISSING VALUES:")
nulls = df.isnull().sum()
print(f"   TotalCharges: {nulls.get('TotalCharges', 0)} null rows → fill with median")

print("\n✅ EDA complete. Proceed to 02_preprocessing.ipynb")
print("=" * 60)